In [ ]:
!pip install transformers sentencepiece accelerate -q

In [ ]:
import pandas as pd
from tqdm import tqdm

from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# LOAD DATA
train_df = pd.read_csv("train.csv")
prompt_df = pd.read_csv("prompting_test.csv")

prompt_df = prompt_df.head(100)

prompt_df["label"] = prompt_df["label"].map({
    "FAKE": 0,
    "REAL": 1
})

In [ ]:
# Load mô hình Prompting

classifier = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    device=-1,                 # CPU
    max_new_tokens=2
)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [ ]:

# ZERO-SHOT PROMPTING
def zero_shot_prompt(text):

    prompt = f"""
REAL or FAKE news?

{text}
"""

    return prompt

# Hàm dự đoán

def predict_zero_shot(text):

    prompt = zero_shot_prompt(text)

    result = classifier(prompt)[0]["generated_text"]

    result = result.strip().upper()

    if "REAL" in result:
        return 1
    else:
        return 0


predictions = []

for text in tqdm(prompt_df["text"]):

    pred = predict_zero_shot(text)

    predictions.append(pred)

# Đánh giá mô hình

y_true = prompt_df["label"]
y_pred = predictions

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1-score :", f1_score(y_true, y_pred))

  0%|          | 0/100 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1953 > 512). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=2) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 100/100 [12:47<00:00,  7.68s/it]

Accuracy : 0.67
Precision: 0.67
Recall   : 1.0
F1-score : 0.8023952095808383
